In [1]:
from utils.util import *
import geopandas as gpd
from shapely.geometry import shape
import folium
import os
import sys
import time
import pandas as pd

def gatherData(dataset, year, city, aoi_geodf, isGroundTruth = False):
    bandNames = {'B2', 'B3', 'B4', 'B5', 'B6', 'ST_B10'}
    if isGroundTruth:
        bandNames = {'ST_B10'}
    includeMetadata = True
    #%%
    # Cell 6: Search for Scenes
    if dataset == 'srtm_v3' and year != 2014:
        return
    search_payload = createSceneSearchPayload(dataset, aoi_geodf, year, 20)
    # print(search_payload)
    scenes = sendRequest(serviceUrl + "scene-search", search_payload, apiKey)
    pd.json_normalize(scenes['results'])
    if len(scenes['results']) == 0:
        return
    #%%
    # Cell 7: Collect Entity IDs
    entityIds = [result['entityId'] for result in scenes['results'] if result['options']['bulk']]
    #%%
    # Cell 8: Prepare Scene List for Download
    listId = f"temp_{dataset}_list"
    scn_list_add_payload = {
        "listId": listId,
        'idField': 'entityId',
        "entityIds": entityIds,
        "datasetName": dataset
    }
    sendRequest(serviceUrl + "scene-list-add", scn_list_add_payload, apiKey)
    #%%
    # Cell 9: Prepare Download Options
    download_opt_payload = {
        "listId": listId,
        "datasetName": dataset,
    }
    products = sendRequest(serviceUrl + "download-options", download_opt_payload, apiKey)
    pd.json_normalize(products)
    
    #%%
    # Cell 10: Collect Files to Download
    downloads = []
    if 'landsat_ot_c2_l2' in dataset:
        for product in products:
            if product["secondaryDownloads"]:
                for secDownload in product["secondaryDownloads"]:
                    if secDownload["bulkAvailable"] and any(band in secDownload['displayId'] for band in bandNames):
                        downloads.append({"entityId": secDownload["entityId"], "productId": secDownload["id"]})
                    if includeMetadata and secDownload['displayId'].endswith('_MTL.txt'):
                        downloads.append({"entityId": secDownload["entityId"], "productId": secDownload["id"]})
    elif 'srtm_v3' in dataset:
        for product in products:
            if product["bulkAvailable"] and product["entityId"] and product["id"]:
                downloads.append({"entityId": product["entityId"], "productId": product["id"]})
    elif 'ccdc_v1_3' in dataset:
        for product in products:
            if product["bulkAvailable"] and product["entityId"] and product["id"]:
                downloads.append({"entityId": product["entityId"], "productId": product["id"]})
    #%%
    # Cell 11: Submit Download Request
    download_req_payload = {
        "downloads": downloads,
        "label": listId
    }
    download_request_results = sendRequest(serviceUrl + "download-request", download_req_payload, apiKey)
    #%%
    print(download_request_results)
    #%%
    # Cell 12: Download Files
    if dataset == 'landsat_ot_c2_l2':
        results = download_request_results['availableDownloads']
        for result in results:
            runDownload(threads, result['url'])
    elif dataset == 'srtm_v3':
        result = download_request_results['preparingDownloads'][2]
        runDownload(threads, result['url'])
    else:
        result = download_request_results['preparingDownloads'][0]
        runDownload(threads, result['url'])
    for _, t in enumerate(threads):
        t.join()
    for file in os.listdir('./Unprocessed'):
        if file.endswith('.tar'):
            try: 
                extract_specific_files('./Unprocessed/' + file, './Unprocessed')
            except:
                print(f'Error: Could not extract file {file}')
    #%%
    # Cell 13: Clean Up
    remove_scnlst_payload = {"listId": listId}
    sendRequest(serviceUrl + "scene-list-remove", remove_scnlst_payload, apiKey)
    #%%
    # Cell 15: Verify Downloads
    import rasterio
    from shapely.geometry import box
    
    usableTIFFs = []
    print(os.listdir(unprocessed_dir))
    # Ensure the shapefile matches the TIF CRS
    for tif_file in os.listdir(unprocessed_dir):
        if tif_file.endswith(".TIF") or tif_file.endswith(".tif"):
            tif_path = unprocessed_dir + '/' + tif_file
            with rasterio.open(tif_path) as src:
                # Reproject shapefile to match TIF file CRS
                aoi_geodf_proj = aoi_geodf.to_crs(src.crs)
                tif_bounds = box(*src.bounds)
                
                # Check containment
                for idx, geom in enumerate(aoi_geodf_proj.geometry):
                    if tif_bounds.contains(geom):
                        usableTIFFs.append(tif_file)
                        print(f"Polygon {city} is fully inside {tif_file}")
                    else:
                        print(f"Polygon {city} is NOT fully inside {tif_file}")
    print(usableTIFFs)
    #%%
    goodCoordinates = clipUnprocessedRasters(usableTIFFs, aoi_geodf_proj)
    #%%
    for file in os.listdir(unprocessed_dir):
        if ".txt" in file or "Clipped_" in file:
            if '1arc_v3' in file:
                moveToRaw(file, 'DEM', f'{year}-01-01', city)
                continue
            if 'LCMAP' in file:
                if 'LCPRI' in file:
                    moveToRaw(file, 'Land_Cover', f'{year}-01-01', city)
                continue
            date, band, coordinate = getMetaFromLandsatTIRs(file)
            if coordinate not in goodCoordinates:
                continue
            print(date, band)
            if band == 'B10':
                moveToRaw(file, 'LST', date, city)
            if isGroundTruth:
                continue
            if band == 'B2':
                moveToRaw(file, 'Albedo', date, city)
            if band == 'B3':
                moveToRaw(file, 'NDWI', date, city)
            if band == 'B4':
                moveToRaw(file, 'Albedo', date, city)
                moveToRaw(file, 'NDVI', date, city)
            if band == 'B5':
                moveToRaw(file, 'Albedo', date, city)
                moveToRaw(file, 'NDVI', date, city)
                moveToRaw(file, 'NDWI', date, city)
            if band == 'B6':
                moveToRaw(file, 'Albedo', date, city)
            if band == 'MTL':
                moveToRaw(file, 'Albedo', date, city)
                moveToRaw(file, 'LST', date, city)
                moveToRaw(file, 'NDVI', date, city)
                moveToRaw(file, 'NDWI', date, city)    
    #%%
    print('Finished moving')
    truthString = "InputData"
    if isGroundTruth:
        truthString = "GroundTruth"
    with open('progress.txt', "a") as file:
        file.write(str(truthString + ":" + str(city) + ":" + str(year) + ":" + dataset + "\n"))
    print('progress written for', truthString, city, year, dataset)

Directory './Unprocessed' already exists.
Directory './Data' already exists.
Directory './RawClippedRasters' already exists.
Directory './Data/LST' already exists.
Directory './Data/NDVI' already exists.
Directory './Data/NDWI' already exists.
Directory './Data/Land_Cover' already exists.
Directory './Data/Albedo' already exists.
Directory './Data/DEM' already exists.
Directory './RawClippedRasters/LST' already exists.
Directory './RawClippedRasters/NDVI' already exists.
Directory './RawClippedRasters/NDWI' already exists.
Directory './RawClippedRasters/Land_Cover' already exists.
Directory './RawClippedRasters/Albedo' already exists.
Directory './RawClippedRasters/DEM' already exists.
Logging in...


Login was unsuccessful, please try again or create an account at: https://ers.cr.usgs.gov/register.


In [2]:
datasets = ['landsat_ot_c2_l2', 'srtm_v3', 'ccdc_v1_3', 'landsat_ot_c2_l2'] 
'''['nlcd_collection_lndcov']'''
years = [year for year in range(2013, 2022)]

# Cell 2: Load shape file
shapefile_folder = "./Data/area_shp/"
shapeGroundTruth_folder = "./Data/label_shp/"
cities = []
aoi_geodfs = []
aoi_geodfsTruth = []
for file in os.listdir(shapefile_folder):
    if file.endswith(".shp"):
        cities.append(file.replace('Polygon_', '').replace('.shp', ''))
        aoi_geodf = gpd.read_file(shapefile_folder + file)
        aoi_geodf = aoi_geodf.to_crs("EPSG:4326")
        if aoi_geodf.empty:
            sys.exit("Error: Shapefile contains no data.")
        aoi_geodfs.append(aoi_geodf)
for file in os.listdir(shapeGroundTruth_folder):
    if file.endswith(".shp"):
        cities.append(file.replace('Polygon_', '').replace('.shp', ''))
        aoi_geodf = gpd.read_file(shapefile_folder + file)
        aoi_geodf = aoi_geodf.to_crs("EPSG:4326")
        if aoi_geodf.empty:
            sys.exit("Error: Shapefile contains no data.")
        aoi_geodfsTruth.append(aoi_geodf)
print("Shapefiles loaded successfully.")
import traceback
for dataset in datasets:
    for year in years[1:4]:
        i = 0
        while i < len(cities[:3]):
            try:
                clear_folder(unprocessed_dir)
                assert len(os.listdir(unprocessed_dir)) == 0, "Unprocessed directory is not empty."
                isGroundTruth = i == len(dataset) - 1
                truthString = "GroundTruth" if isGroundTruth else "InputData"
                city, aoi_geodf = cities[i], aoi_geodfsTruth[i] if isGroundTruth else aoi_geodfs[i]
                print(f'Gathering {truthString} {dataset} for {year} in {city}.')
                if os.path.exists('progress.txt'):
                    with open('progress.txt', 'r') as file:
                        progress = [line.split(':') for line in file.read().strip().split('\n')]
                    if any(truthString == instance[0] and city == instance[1] and str(year) == instance[2] and dataset == instance[3] for instance in progress):
                        print(f"{truthString}, {city}, {year}, {dataset} was gathered in the past.")
                    else:
                        gatherData(dataset, year, city, aoi_geodf, isGroundTruth)
                i += 1
            except Exception as e:
                print("An exception occurred:")
                print(f"Exception: {e}")
                traceback.print_exc()  # Print the full stack trace
                time.sleep(60)
print("Gathered data successfully.")

Shapefiles loaded successfully.
Gathering landsat_ot_c2_l2 for 2014 in Abilene_TX.
Expecting value: line 1 column 1 (char 0)


AttributeError: 'tuple' object has no attribute 'tb_frame'